# Notebook 04: autoencoders and one-class deep models on KDD99

**Class 03_02.** Same protocol as notebook 02 (train on normal rows, threshold at 1% false alarms on normal
validation rows, test with 1.8% attacks), now with neural detectors in Keras 3 on the JAX backend:

| model | trained to | anomaly score |
|:--|:--|:--|
| autoencoder (AE), one or several hidden layers | reconstruct $\mathbf{x}$ through a narrow code $\mathbf{h}$ | $\lVert\mathbf{x}-\hat{\mathbf{x}}\rVert^2/d$ |
| residual AE | same, each hidden block computes $\mathbf{h}+F(\mathbf{h})$ (skip connection) | same |
| sparse AE | same + $\lambda\lVert\mathbf{h}\rVert_1$ (few active code units) | same |
| denoising AE | reconstruct $\mathbf{x}$ from $\mathbf{x}+\boldsymbol\epsilon$ | same |
| variational AE | reconstruct + $\beta\,\mathrm{KL}(q(\mathbf{h}\mid\mathbf{x})\Vert\mathcal{N}(0,I))$ | same |
| Deep SVDD | map normal rows close to a centre $\mathbf{c}$ | $\lVert\phi(\mathbf{x})-\mathbf{c}\rVert^2$ |

A **linear** AE with a mean-squared loss learns the same subspace as PCA (Baldi and Hornik 1989): the PCA
reconstruction detector of notebook 02 is the AE's twin and its baseline.

All AEs are built by one function, `build_autoencoder` in `src/autoenc.py` (depth, activation, normalisation,
residual blocks, sparsity, input noise, and a temporal body for notebook 05). Section D asks the question that the
one-class setting makes hard: **where do we put the threshold without labels?**

In [ ]:
import os
import sys
import time

os.environ.setdefault("KERAS_BACKEND", "jax")
import keras
import numpy as np
import polars as pl
from matplotlib import pyplot as plt

sys.path[:0] = [".", "../../src"]  # this folder and the generic libraries of the repository
import aasplot as ap
import anomlib as an
import autoenc
import deeplib as dl
import thresholds as th
from aasplot import BLUE, COL, GOLD, GREEN, GREY, LIGHT, RED

an.table_config()
ap.setup("03")
print("Keras", keras.__version__, "backend", keras.backend.backend())
sp = an.kdd_split(seed=0)
Xtr, Xva, Xte, y = sp.X("train"), sp.X("val"), sp.X("test"), sp.y_test
cats = sp.test["category"]

## A. The plain autoencoder, step by step

71 inputs $\to$ 32 (SiLU) $\to$ 8 (code) $\to$ 32 (SiLU) $\to$ 71, built by `build_autoencoder((71,), widths=(32,),
code=8)` and trained with Adam on the mean-squared error of *normal* rows only; 10% of them are kept aside and
early stopping restores the best epoch. SiLU, $x\,\sigma(x)$, is a smooth ReLU that ASAP 2.0 (Silva et al. 2025)
also uses.

In [ ]:
model = autoenc.build_autoencoder((Xtr.shape[1],), widths=(32,), code=8)
model.summary()
hist = autoenc.fit_autoencoder(model, Xtr, epochs=60, seed=0).history
f, ax = ap.fig("full", 1.4)
ax.plot(hist["loss"], color=BLUE, label="train")
ax.plot(hist["val_loss"], color=RED, label="validation (normal rows)")
ax.set(xlabel="epoch", ylabel="MSE", yscale="log")
ax.legend()
plt.show()
ae = dl.AE(latent=8, widths=(32,), seed=0).fit(Xtr)  # the same model as a Detector: fit / score
s_va, s_te = ae.score(Xva), ae.score(Xte)
t = an.threshold(s_va)

In [ ]:
def ae_scores(s_normal, s_attack, threshold: float) -> None:
    """Histogram of the AE score on the test set: normal rows against attacks, with the 1% false-alarm threshold."""
    f, ax = ap.fig("full", 1.3)
    allv = np.r_[s_normal, s_attack]
    bins = np.logspace(np.log10(allv.min() + 1e-6), np.log10(allv.max()), 55)
    ax.hist(s_normal, bins, density=True, color=BLUE, alpha=0.65, label="normal", lw=0)
    ax.hist(s_attack, bins, density=True, color=RED, alpha=0.65, label="attack", lw=0)
    ax.axvline(threshold, color=GREY, ls="--", lw=1)
    ax.text(threshold * 1.15, ax.get_ylim()[1] * 0.85, "threshold\n(1% false alarms)", color=GREY, fontsize=7, va="top")
    ax.set(xscale="log", xlabel="reconstruction error = anomaly score", ylabel="density", yticks=[])
    ax.legend(loc="upper left")
    ax.grid(False)
    ap.save(f, "ae_scores")


ae_scores(s_te[y == 0], s_te[y == 1], t)

**Why not a linear code?** On data that lie on a curve a one-unit linear code (PCA) cannot follow the curve, a
non-linear one-unit AE can. This is the case for depth: it models structure that a plane misses.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.neural_network import MLPRegressor


def ae_vs_pca() -> None:
    """Toy data on a curved one-dimensional structure: a linear code (PCA, 1 unit) against an AE with a 1-unit code."""
    rng = np.random.default_rng(0)
    t = rng.uniform(-1.3, 1.3, 400)
    X = np.c_[t, 0.9 * t**2 - 0.4] + rng.normal(0, 0.04, (400, 2))
    pc = PCA(1).fit(X)
    lin = pc.inverse_transform(pc.transform(X))
    ae = MLPRegressor(hidden_layer_sizes=(16, 1, 16), activation="tanh", max_iter=4000, random_state=0, tol=1e-6).fit(
        X, X
    )
    rec = ae.predict(X)
    e_lin, e_ae = float(((X - lin) ** 2).sum(1).mean()), float(((X - rec) ** 2).sum(1).mean())
    f, axs = plt.subplots(1, 2, figsize=(COL * 2 + 0.1, 1.9), sharey=True, gridspec_kw={"wspace": 0.08})
    for ax, r, col, name, e in (
        (axs[0], lin, BLUE, "linear code: PCA", e_lin),
        (axs[1], rec, RED, "AE: code of 1 unit, tanh layers", e_ae),
    ):
        ax.scatter(X[:, 0], X[:, 1], s=3, color=GREY, alpha=0.5, lw=0)
        o = np.argsort(r[:, 0])
        ax.plot(r[o, 0], r[o, 1], color=col, lw=1.8)
        ax.set(title=name, xticks=[], yticks=[], aspect="equal")
        ax.set_xlabel(f"mean squared error {e:.3f}", fontsize=7)
        ax.grid(False)
    ap.save(f, "ae_vs_pca")
    print("ae_vs_pca mse: PCA", round(e_lin, 4), "AE", round(e_ae, 4))


ae_vs_pca()

## B. Deeper, residual and the other deep detectors (3 seeds)

`dl.AE` wraps `build_autoencoder`. The variants below change **one** thing from the plain AE (one hidden layer of
32 units, code 8): more hidden layers (a funnel 64-32 and 128-64-32 into the same code), a residual skip in every
block with layer normalisation, sparsity ($L_1$ on the code), input noise (denoising), a variational code, and a
one-class sphere (Deep SVDD). PCA and the Gaussian mixture are the classic references.

In [ ]:
rows = []
for seed in [0, 1, 2]:
    dets = [dl.AE(seed=seed), dl.AE(widths=(64, 32), seed=seed), dl.AE(widths=(128, 64, 32), seed=seed),
            dl.AE(widths=(64, 32), residual=True, norm="layer", seed=seed), dl.AE(l1=1e-4, seed=seed),
            dl.AE(noise=0.1, seed=seed), dl.VAE(seed=seed), dl.DeepSVDD(seed=seed), an.PCARecon(), an.GMM(seed=seed)]  # fmt: skip
    for det in dets:
        t0 = time.time()
        det.fit(Xtr)
        s_va, s_te = det.score(Xva), det.score(Xte)
        rows.append({"seed": seed, "detector": det.name, **an.evaluate(s_te, y, s_va),
                     **an.recall_by_category(s_te, cats, s_va), "seconds": time.time() - t0})  # fmt: skip
deep = pl.DataFrame(rows)
cols = ["AUROC", "AUPRC", "F1", "dos", "probe", "r2l", "u2r", "seconds"]
print(deep.group_by("detector").agg([pl.col(c).mean() for c in cols] + [pl.col("AUPRC").std().alias("AUPRC std")])
      .sort("AUPRC", descending=True))  # fmt: skip

In [ ]:
SHORT = {
    "GMM (-log-likelihood)": "GMM",
    "PCA reconstruction": "PCA (linear twin)",
    "PCA recon (space)": "PCA reconstruction",
    "predict from others (space)": "predict from the others",
    "conv AE (space+time)": "conv AE (space + time)",
    "Isolation Forest (space)": "Isolation Forest",
    "dense AE (space)": "dense AE",
    "per-sensor max|z|": "per-sensor max |z|",
}


def short(name: str) -> str:
    return SHORT.get(name, name)


def deep_band(rows) -> None:
    """rows: one dict per (seed, detector) with AUPRC (3 seeds)."""
    acc: dict = {}
    for r in rows:
        acc.setdefault(r["detector"], []).append(r["AUPRC"])
    data = sorted(((np.mean(v), np.std(v, ddof=1), k) for k, v in acc.items()), reverse=True)
    f, ax = ap.fig("full", 1.35)
    ys = np.arange(len(data))[::-1]
    cols = [GOLD if "GMM" in k else (GREEN if "PCA" in k else BLUE) for _, _, k in data]
    ax.barh(
        ys,
        [m for m, _, _ in data],
        xerr=[s for _, s, _ in data],
        color=cols,
        height=0.62,
        error_kw={"ecolor": RED, "lw": 1.1, "capsize": 2},
    )
    for y, (m, _, _) in zip(ys, data):
        ax.text(0.405, y, f"{m:.3f}", va="center", ha="left", fontsize=7, color="white")
    ax.set_yticks(ys, [short(k) for _, _, k in data])
    ax.set(xlabel="AUPRC, mean ± std over 3 seeds (gold: best classic, green: linear twin)", xlim=(0.4, 0.75))
    ax.grid(axis="y", visible=False)
    ap.save(f, "deep_band")


deep_band(deep.to_dicts())

**Reading.** Depth is what moves the needle: a second hidden layer lifts the AE from AUPRC 0.58 to 0.72, above the
Gaussian mixture of notebook 02 (0.68) and far above the linear twin (PCA, 0.55). A third layer adds nothing
(0.71), and the residual blocks with layer normalisation (0.71) cost four times the training time for no gain:
skip connections matter for deep stacks, not for two layers. Sparsity, a denoising input and a variational code
*hurt* here (0.54 to 0.59): they regularise a model that was not over-fitting. PCA keeps the best F1 and the best
r2l recall. The ranking is the one to expect on tabular data (ADBench, Han et al. 2022): no family wins by default.

## C. How big should the code be?

In [ ]:
rows = []
for latent in [2, 4, 8, 16, 32, 64, 128]:
    det = dl.AE(latent=latent, widths=(max(32, 2 * latent),), seed=0).fit(Xtr)
    rows.append({"code size": latent, **an.evaluate(det.score(Xte), y, det.score(Xva)),
                 "val MSE": det.history.history["val_loss"][-1]})  # fmt: skip
print(pl.DataFrame(rows))

In [ ]:
from matplotlib.ticker import ScalarFormatter


def code_size(rows, gmm_auprc: float) -> None:
    """rows: dicts with `code size`, AUPRC and `val MSE` (validation MSE of normal rows)."""
    rows = sorted(rows, key=lambda r: r["code size"])
    x = [r["code size"] for r in rows]
    f, ax = ap.fig("full", 1.5)
    ax.plot(x, [r["AUPRC"] for r in rows], color=BLUE, marker="o", ms=3.5, label="AUPRC (test attacks)")
    ax.axhline(gmm_auprc, color=GOLD, ls="--", lw=1)
    ax.text(128, gmm_auprc + 0.012, f"GMM {gmm_auprc:.3f}", color=GOLD, fontsize=7, ha="right")
    best = max(rows, key=lambda r: r["AUPRC"])
    ax.annotate(
        f"{best['AUPRC']:.3f}: picked by looking at the test attacks",
        (best["code size"], best["AUPRC"]),
        (2.2, 0.745),
        fontsize=7,
        color=BLUE,
        arrowprops={"arrowstyle": "-", "color": BLUE, "lw": 0.6},
    )
    ax.set(xscale="log", xlabel="code size", ylabel="AUPRC", ylim=(0.4, 0.78))
    ax.set_xscale("log", base=2)
    ax.set_xticks(x, [str(v) for v in x])
    ax.xaxis.set_minor_formatter(ScalarFormatter())
    ax2 = ax.twinx()
    ax2.plot(
        x, [r["val MSE"] for r in rows], color=GREY, marker="s", ms=3, ls=":", label="validation MSE (normal only)"
    )
    ax2.set(ylabel="validation MSE")
    ax2.grid(False)
    ax2.spines["right"].set_visible(True)
    h1, l1 = ax.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax.legend(h1 + h2, l1 + l2, loc="lower center", ncol=2, fontsize=6.8)
    ap.save(f, "code_size")


code_size(rows, float(deep.filter(pl.col("detector").str.starts_with("GMM"))["AUPRC"].mean()))

Too small a code (2, 4) cannot hold normal traffic, so normal rows also reconstruct badly; too large a code (128,
wider than the 71 inputs) drifts towards copying everything, attacks included. In between, a 32-unit code is the
best detector of the whole class (AUPRC 0.71, above the GMM's 0.68).

**But who chose 32?** We picked it by looking at the *test attacks*. In the one-class setting there are no labelled
attacks to tune on, and the validation MSE of normal data (last column) prefers 64, not 32. Every hyper-parameter
of an anomaly detector is a guess unless some labelled anomalies (or a realistic simulation of them) exist; a
reported gain from tuning on the test set is not a gain.

## D. Where is the threshold? Without labels, with ISODATA

Until now the threshold was the 99% quantile of the scores on **normal validation rows**: it needs clean data and
fixes the false-alarm rate by decree. ASAP 2.0 (Silva et al. 2025) instead lets the reconstruction errors of
unlabelled traffic choose: **ISODATA** (intermeans, Ridler and Calvard 1978) starts at the mean of all errors and
repeats $T\leftarrow\tfrac12(\bar e_{\le T}+\bar e_{>T})$ until $T$ stops moving. No label is used, but it
assumes **two modes** (normal low, anomalous high). Our test set has 1.8% attacks, the paper's was balanced by
undersampling. We vary the attack share and compare four rules on the same AE (2 hidden layers, SiLU):
the quantile of normal validation scores, ISODATA on the raw error, ISODATA on $\log e$ (errors are heavy-tailed),
and the best possible F1 (an *oracle* that looks at the labels).

In [ ]:
best = dl.AE(widths=(64, 32), seed=0).fit(Xtr)
s_va, s_te = best.score(Xva), best.score(Xte)
ia, i0 = np.flatnonzero(y == 1), np.flatnonzero(y == 0)
RULES = {"quantile 99% (normal validation)": lambda s, yy: th.quantile(s_va, 0.01),
         "ISODATA, raw error": lambda s, yy: th.isodata(s),
         "ISODATA, log error": lambda s, yy: th.isodata(s, log=True),
         "oracle (best F1)": lambda s, yy: th.oracle_f1(s, yy)}  # fmt: skip


def at_share(share: float, draw: int):
    """The attacks of the test set mixed with enough normal rows to make `share` of the sample attacks."""
    n0 = min(int(len(ia) * (1 - share) / share), len(i0))
    idx = np.r_[ia, np.random.default_rng(draw).choice(i0, n0, replace=False)]
    return s_te[idx], y[idx]


rows = []
for share in [0.018, 0.05, 0.1, 0.2, 0.3, 0.5]:
    for draw in range(5):
        s, yy = at_share(share, draw)
        for rule, pick in RULES.items():
            rows.append({"attack share": share, "rule": rule, **an.metrics_at(s, yy, pick(s, yy))})
thr = pl.DataFrame(rows)
print(thr.group_by(["attack share", "rule"], maintain_order=True).agg(pl.col(["F1", "MCC"]).mean().round(2)))

In [ ]:
def thr_base_rate(thr) -> None:
    """F1 of four threshold rules against the share of attacks in the unlabelled sample (mean of 5 draws)."""
    f, ax = ap.fig("full", 1.8)
    style = {"quantile 99% (normal validation)": (BLUE, "o", "-"), "ISODATA, raw error": (GOLD, "s", "--"),
             "ISODATA, log error": (RED, "^", "-"), "oracle (best F1)": (GREY, None, ":")}  # fmt: skip
    for rule, (col, mk, ls) in style.items():
        d = thr.filter(pl.col("rule") == rule).group_by("attack share").agg(pl.col("F1").mean()).sort("attack share")
        ax.plot(d["attack share"] * 100, d["F1"], color=col, marker=mk, ms=3.5, ls=ls, label=rule)
    ax.set(xscale="log", xlabel="attacks in the unlabelled sample (%)", ylabel="F1 at the threshold", ylim=(0, 1.02))
    ax.set_xticks([1.8, 5, 10, 20, 50], ["1.8", "5", "10", "20", "50"])
    ax.minorticks_off()
    ax.legend(loc="lower right", fontsize=6.3)
    ap.save(f, "thr_base_rate")


thr_base_rate(thr)

In [ ]:
def thr_hist(panels) -> None:
    """Score histograms with the thresholds of the rules for two attack shares. panels: [(share, draw)]."""
    f, axs = ap.fig("full", 1.45, ncols=2, sharey=True, gridspec_kw={"wspace": 0.08})
    for ax, (share, draw) in zip(axs, panels):
        s, yy = at_share(share, draw)
        bins = np.logspace(np.log10(s.min()), np.log10(s.max()), 50)
        ax.hist(s[yy == 0], bins, color=BLUE, alpha=0.65, lw=0, label="normal")
        ax.hist(s[yy == 1], bins, color=RED, alpha=0.65, lw=0, label="attack")
        for (rule, pick), col, ls in zip(RULES.items(), [BLUE, GOLD, RED, GREY], ["--", "--", "-", ":"]):
            ax.axvline(pick(s, yy), color=col, ls=ls, lw=1.1)
        ax.set(xscale="log", xlabel="reconstruction error", title=f"{share:.1%} attacks".replace(".0%", "%"), yticks=[])
        ax.grid(False)
    axs[0].legend(fontsize=6.5, loc="upper left")
    ap.save(f, "thr_hist")


thr_hist([(0.018, 0), (0.3, 0)])

**Reading.** At 1.8% attacks ISODATA on the log error fails (F1 0.12 against 0.60 for the quantile): the second mode
is too small to move the means and the cut lands inside the normal errors. From 10% attacks it works (0.70, then
0.86 at 20%, 0.90 at 50%) and follows the oracle (0.83, 0.87, 0.96) without a single label; above 20% it even beats the
fixed 1% quantile, which keeps its false-alarm rate while the base rate rises. On the raw error ISODATA never
works (F1 0.45 at every share): the heavy tail of the errors drags the means up, so **take the logarithm first**.
The quantile rule is stable because it never looks at the mixture, but it needs clean validation rows and decides
the false-alarm rate instead of finding the boundary. The paper's results come from a balanced test set; in a real
network use the quantile, and use ISODATA to **monitor** a stream or a campaign where attacks are frequent.

## E. Explaining an alarm: the error per feature

In [ ]:
names = np.array(sp.prep.names)
err = (ae.model.predict(Xte, verbose=0) - Xte) ** 2
tops = {}
for c in an.CATEGORIES:
    e = err[(cats == c).to_numpy()].mean(0)
    top = np.argsort(e)[::-1][:4]
    tops[c] = [(names[i], float(e[i])) for i in top]
    print(f"{c:6}", ", ".join(f"{names[i]} ({e[i]:.1f})" for i in top))

In [ ]:
def explain_alarm(tops: dict) -> None:
    """tops: {category: [(feature, mean squared error), ...]} the largest per-feature errors of the AE on test attacks."""
    f, axs = ap.fig("full", 1.35, ncols=len(tops), gridspec_kw={"wspace": 0.12})
    for ax, (cat, items) in zip(axs, tops.items()):
        names, vals = zip(*items)
        ys = np.arange(len(vals))[::-1]
        ax.barh(ys, vals, color=LIGHT, height=0.7)
        for y, n, v in zip(ys, names, vals):
            ax.text(0.02 * max(vals), y, n.replace("_", " "), va="center", fontsize=6.3, color="#2C3E50")
        ax.set(title=cat, yticks=[], xticks=[], xlim=(0, max(vals) * 1.25))
        ax.set_xlabel("error")
        ax.grid(False)
    ap.save(f, "explain_alarm")


explain_alarm({c: tops[c] for c in ("dos", "probe", "r2l")})

The features that the AE cannot reconstruct name the reason of the alarm (SYN-error rates for the floods, failed
logins and root shells for r2l / u2r): a first triage hint for the analyst.